# Seasonal Agriculture Performance Analysis

**VOIS AICTE Batch 1 (2026–2027) — Major Project**

This notebook analyzes the provided agricultural dataset to investigate seasonal differences in agricultural performance, environmental conditions, resource usage and economic outcomes.

### Dataset
- Records: 4,000
- Variables: 28
- Seasons: Kharif, Rabi, Zaid
- Missing numeric values are handled using median imputation.
- The original CSV is kept unchanged; the cleaned data is created inside the notebook.

### Problem statement
Agricultural performance can differ across seasons because of environmental conditions, farming practices, resource availability and market conditions. The objective is to identify meaningful seasonal patterns, relationships, differences and variations in the supplied data.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import kruskal

pd.set_option("display.max_columns", None)

df = pd.read_csv("seasonal_agriculture_performance_dataset.csv")
print("Dataset shape:", df.shape)
display(df.head())


## 1. Data quality and cleaning

In [ ]:
missing_before = int(df.isna().sum().sum())
missing_by_column = df.isna().sum().sort_values(ascending=False)
print("Total missing values:", missing_before)
display(missing_by_column[missing_by_column > 0])

clean = df.copy()
numeric_cols = clean.select_dtypes(include=np.number).columns
for col in numeric_cols:
    clean[col] = clean[col].fillna(clean[col].median())

print("Missing values after median imputation:", int(clean.isna().sum().sum()))


### Cleaning decision
The supplied data contains missing values in `Rainfall_mm`, `Soil_Moisture_pct` and `Yield_Tonnes_Ha`. Because these are numeric measurements, median imputation is used to retain all 4,000 records while reducing the influence of extreme observations. No rows are removed solely because of missing values.

## 2. Exploratory analysis

In [ ]:
print("Seasons:", clean["Season"].unique())
print("Crops:", clean["Crop"].nunique())
print("States:", clean["State"].nunique())
print("Irrigation methods:", clean["Irrigation_Method"].unique())

season_summary = clean.groupby("Season").agg(
    Farms=("Farm_ID","count"),
    Avg_Yield_Tonnes_Ha=("Yield_Tonnes_Ha","mean"),
    Avg_Production_Tonnes=("Production_Tonnes","mean"),
    Avg_Revenue_INR=("Revenue_INR","mean"),
    Avg_Cost_INR=("Total_Cost_INR","mean"),
    Avg_Profit_INR=("Profit_INR","mean"),
    Avg_Water_Used_m3=("Water_Used_m3","mean"),
    Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3","mean"),
    Avg_Disease_Pest_Risk_pct=("Disease_Pest_Risk_pct","mean"),
    Avg_Rainfall_mm=("Rainfall_mm","mean"),
    Avg_Temperature_C=("Avg_Temperature_C","mean"),
).round(2)

display(season_summary)


### Average yield by season

![Average yield by season](charts/01_average_yield_by_season.png)

### Average profit by season

![Average profit by season](charts/02_average_profit_by_season.png)

### Seasonal environmental conditions

![Seasonal environmental conditions](charts/03_seasonal_environment.png)

## 3. Crop-level analysis

In [ ]:
crop_summary = clean.groupby("Crop").agg(
    Farms=("Farm_ID","count"),
    Avg_Yield_Tonnes_Ha=("Yield_Tonnes_Ha","mean"),
    Avg_Profit_INR=("Profit_INR","mean"),
    Avg_Revenue_INR=("Revenue_INR","mean"),
    Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3","mean"),
    Avg_Risk_pct=("Disease_Pest_Risk_pct","mean")
).round(2)

display(crop_summary.sort_values("Avg_Profit_INR", ascending=False))


### Average profit by crop

![Average profit by crop](charts/04_average_profit_by_crop.png)

## 4. Resource and irrigation analysis

In [ ]:
irrigation_summary = clean.groupby("Irrigation_Method").agg(
    Farms=("Farm_ID","count"),
    Avg_Water_Used_m3=("Water_Used_m3","mean"),
    Avg_Water_Efficiency=("Water_Efficiency_t_per_1000m3","mean"),
    Avg_Yield_Tonnes_Ha=("Yield_Tonnes_Ha","mean"),
    Avg_Profit_INR=("Profit_INR","mean")
).round(2)

display(irrigation_summary.sort_values("Avg_Water_Efficiency", ascending=False))


### Water efficiency by irrigation method

![Water efficiency by irrigation method](charts/05_water_efficiency_irrigation.png)

## 5. Crop × season comparison

In [ ]:
season_crop_profit = clean.pivot_table(
    index="Crop", columns="Season", values="Profit_INR", aggfunc="mean"
).round(0)

display(season_crop_profit)


### Average profit by crop and season

![Average profit by crop and season](charts/06_crop_season_profit_heatmap.png)

## 6. Relationships with profit

In [ ]:
corr_profit = clean.select_dtypes(include=np.number).corr()["Profit_INR"].sort_values(ascending=False)
display(corr_profit.to_frame("Pearson_correlation_with_profit"))


### Variables most associated with profit

![Variables most associated with profit](charts/07_profit_correlations.png)

## 7. Profitability by season

In [ ]:
profit_rate = clean.groupby("Season").agg(
    Profitable_Farms_pct=("Profit_INR", lambda s: (s > 0).mean()*100),
    Median_Profit_INR=("Profit_INR","median"),
    Median_Yield_Tonnes_Ha=("Yield_Tonnes_Ha","median")
).round(2)

display(profit_rate)


### Share of profitable farms by season

![Share of profitable farms by season](charts/08_profitable_farms_by_season.png)

## 8. State-level context

In [ ]:
state_summary = clean.groupby("State").agg(
    Avg_Profit_INR=("Profit_INR","mean"),
    Avg_Yield_Tonnes_Ha=("Yield_Tonnes_Ha","mean")
).sort_values("Avg_Profit_INR", ascending=False).round(2)

display(state_summary)


### Average profit by state

![Average profit by state](charts/09_average_profit_by_state.png)

## 9. Statistical comparison across seasons

In [ ]:
yield_groups = [g["Yield_Tonnes_Ha"].values for _, g in clean.groupby("Season")]
profit_groups = [g["Profit_INR"].values for _, g in clean.groupby("Season")]

yield_test = kruskal(*yield_groups)
profit_test = kruskal(*profit_groups)

print("Yield Kruskal-Wallis statistic:", round(yield_test.statistic, 4))
print("Yield p-value:", yield_test.pvalue)
print("Profit Kruskal-Wallis statistic:", round(profit_test.statistic, 4))
print("Profit p-value:", profit_test.pvalue)


### Interpretation
The Kruskal–Wallis test checks whether the distributions differ across the three seasons. The very small p-values in this dataset indicate statistically detectable differences in both yield and profit distributions across seasons. This is an association in the supplied data, not proof that season alone causes the observed differences.

## 10. Key findings and evidence-based recommendations

1. **Seasonal variation is visible:** Kharif has the highest average yield and average profit in the supplied dataset, while Zaid has the lowest averages for both.
2. **Profitability differs by season:** the share of farms with positive profit is higher in Kharif than in Rabi and Zaid.
3. **Environmental conditions vary strongly:** Kharif has substantially higher average rainfall, while Zaid has the highest average temperature and lower rainfall.
4. **Crop performance is heterogeneous:** crop-level yield, revenue and profit vary considerably, so seasonal analysis should be combined with crop context.
5. **Water efficiency differs by irrigation method:** the dataset shows meaningful differences in water efficiency across irrigation categories.
6. **Profit is strongly associated with revenue and production:** correlation analysis shows that revenue and production have positive relationships with profit in this dataset.
7. **Recommendations:** use seasonal performance dashboards for planning; examine high-variation crop/season combinations more closely; track water efficiency alongside yield; and use local/state-level comparisons before making operational decisions.
8. **Important limitation:** these results describe the supplied dataset. They should not be treated as causal conclusions or as universal agricultural recommendations without additional field, temporal and market data.


## 11. Conclusion

The analysis demonstrates that agricultural performance varies across seasons in the supplied dataset, with differences in yield, profitability, environmental conditions, water usage and crop outcomes. Combining descriptive statistics, visualizations, correlation analysis and a non-parametric seasonal comparison provides an evidence-based view of these differences. The completed notebook can be extended with time-series data, weather forecasts, market-price data and predictive models in future work.